# ML Workflow with B2 Cloud Storage

End-to-end machine learning workflow using `jupyter-b2` for data and model management:

1. Load training data from B2
2. Train a model
3. Save model + metrics to B2
4. Share results with team

**Prerequisites:** Authenticate once with the B2 CLI for zero-config auth:
```bash
pip install b2
b2 account authorize <your-key-id> <your-application-key>
```

After that, `%load_ext jupyter_b2` will auto-authenticate every time!

In [ ]:
# Load extension — auto-authenticates from B2 CLI credentials
%load_ext jupyter_b2

## Step 1: Load Training Data from B2

In [ ]:
# Browse available datasets
%b2 ls my-ml-bucket/datasets/

In [ ]:
# Load directly into pandas — auto-detects parquet format
train_df = %b2_load my-ml-bucket/datasets/train.parquet
test_df = %b2_load my-ml-bucket/datasets/test.parquet

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
train_df.head()

## Step 2: Train the Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd

# Prepare features
target_col = 'target'
feature_cols = [c for c in train_df.columns if c != target_col]

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

# Train
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

## Step 3: Save Model + Metrics to B2

In [ ]:
# Save metrics as a DataFrame
from datetime import datetime

metrics = pd.DataFrame([{
    'model': 'RandomForest',
    'accuracy': accuracy,
    'f1_score': f1,
    'n_estimators': 100,
    'train_samples': len(train_df),
    'test_samples': len(test_df),
    'timestamp': datetime.now().isoformat(),
}])

In [ ]:
%%b2_save my-ml-bucket/experiments/rf-v1/metrics.parquet
metrics

In [ ]:
# Save model weights
import joblib
joblib.dump(model, '/tmp/model.joblib')
%b2 upload /tmp/model.joblib my-ml-bucket/experiments/rf-v1/model.joblib

In [ ]:
# Save feature importance
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

In [ ]:
%%b2_save my-ml-bucket/experiments/rf-v1/feature_importance.csv
importance_df

## Step 4: Share Results

In [ ]:
# Browse what we saved
%b2 ls my-ml-bucket/experiments/rf-v1/

In [ ]:
# Generate shareable links for the team
%b2 presign my-ml-bucket/experiments/rf-v1/metrics.parquet --expires 86400